# Report Generator

> **Purpose**: Demonstrate the `reports.py` module — generate formatted Excel and JSON reports from real pipeline outputs.

**All business logic lives in `../reports.py`. This notebook only imports and calls those functions.**

### Reports generated
| Report | Format | Contents |
|--------|--------|---------|
| `audit_report_<ts>.xlsx` | Excel | 6 sheets: Cleaned_Data, Validation, Business_Rules, Anomalies, Quality_Scores, Statistics |
| `pipeline_report_<ts>.json` | JSON | timestamp, pipeline timing, validation summary, business rules, anomaly summary, scores, statistics |

### Excel improvements in this version
- Bold, navy header row on every sheet
- Auto-fit column widths
- Frozen header row (stays visible while scrolling)
- `Percentage` column in Validation and Business_Rules sheets
- `Sample_Indices` (max 10) replaces large index arrays in Business_Rules

> **Previous step**: `scoring.ipynb` | **Full pipeline**: `run_pipeline.ipynb`

In [ ]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from cleaning import load_dataset, run_cleaning
from validation import run_validation
from rules import run_business_rules
from statistics import run_statistics
from anomaly import run_ml_anomalies
from scoring import run_scoring
from reports import export_excel_report, export_json_report, run_reports

DATA_PATH   = os.path.join("..", "data", "dataset_ecommerce_transactions_data.csv")
REPORTS_DIR = os.path.join("..", "reports")
print(f"Reports will be saved to: {os.path.abspath(REPORTS_DIR)}")

## Step 1 — Run the Full Pipeline

In [ ]:
df_raw   = load_dataset(DATA_PATH)
df_clean = run_cleaning(df_raw)

val_result     = run_validation(df_clean)
violations     = run_business_rules(df_clean)
stats_result   = run_statistics(df_clean)
anomaly_result = run_ml_anomalies(df_clean)
scores         = run_scoring(
    df_clean,
    val_result=val_result,
    violations=violations,
    anomaly_result=anomaly_result,
)

print(f"Pipeline complete. Dataset score: {scores['dataset']['dataset_score']}")
print(f"Rules quality score : {scores['dataset']['rules_quality_score']}")
print(f"Anomaly penalty     : {scores['dataset']['anomaly_penalty']}%")

## Step 2 — Generate Both Reports

In [ ]:
# Minimal pipeline_results dict for the JSON timing section
pipeline_results = {
    "status": "success",
    "pipeline_start": "",
    "pipeline_end": "",
    "stages": {
        "cleaning":          {"rows_out": len(df_clean), "elapsed_sec": 0, "error": None},
        "validation":        {"total_violations": len(val_result["violations"]), "elapsed_sec": 0, "error": None},
        "business_rules":    {"violation_types": len(violations), "elapsed_sec": 0, "error": None},
        "anomaly_detection": {"consensus_anomalies": anomaly_result.get("consensus_anomalies", 0), "elapsed_sec": 0, "error": None},
        "scoring":           {"dataset_score": scores["dataset"]["dataset_score"], "elapsed_sec": 0, "error": None},
    },
}

report_paths = run_reports(
    df_clean,
    violations=violations,
    scores=scores,
    val_result=val_result,
    anomaly_result=anomaly_result,
    stats_result=stats_result,
    results=pipeline_results,
    output_dir=REPORTS_DIR,
)

print("=== Reports Generated ===")
print(json.dumps(report_paths, indent=2))

## Step 3 — Preview Excel Sheets

In [ ]:
excel_path = report_paths["exported"]["excel"]
xl = pd.ExcelFile(excel_path)
print("Sheet names:", xl.sheet_names)

print("\n=== Business_Rules Sheet ===")
pd.read_excel(excel_path, sheet_name="Business_Rules")

In [ ]:
print("=== Quality_Scores Sheet ===")
pd.read_excel(excel_path, sheet_name="Quality_Scores")

In [ ]:
print("=== Anomalies Sheet ===")
pd.read_excel(excel_path, sheet_name="Anomalies")

## Step 4 — Preview JSON Report Structure

In [ ]:
json_path = report_paths["exported"]["json"]
with open(json_path, encoding="utf-8") as f:
    json_report = json.load(f)

print("Top-level keys:", list(json_report.keys()))
print("\n=== Pipeline Execution Summary ===")
print(json.dumps(json_report["pipeline_execution_summary"], indent=2))

print("\n=== Validation Summary ===")
print(json.dumps(json_report["validation_summary"], indent=2))

print("\n=== Business Rules Summary ===")
# Show summary without the large violations list
br = {k: v for k, v in json_report["business_rules_summary"].items() if k != "violations"}
print(json.dumps(br, indent=2))

---
## Key Takeaways

- Excel reports have **bold navy headers**, **auto-fit columns**, and **frozen first row** for professional readability
- The JSON `pipeline_execution_summary` records stage-by-stage timing — useful for performance analysis
- Business rules violations in both Excel and JSON now show `percentage` + `sample_indices` (max 10) instead of large index arrays
- The JSON `validation_summary` includes `pass_rate` — quick health check at a glance
- Both report types are timestamped so you can track quality trends over time
- To run the full pipeline automatically (including timing), use `run_pipeline.ipynb` or `python main.py`